In [1]:
import feedparser 
import requests
import re
import pandas as pd
import time
import os

In [2]:
def get_RSS(url):
    return feedparser.parse(url)

In [3]:
def get_CVE_list(url):
    response = requests.get(url + "json/") 
    data = response.json()

    cve_pattern = r"CVE-\d{4}-\d{4,7}" 
    cve_list = list(set(re.findall(cve_pattern, str(data))))

    return cve_list

In [4]:
def get_ID_ANSSI_from_link(url):
    if url[30] == "v":
        return url[34:54]
    else:
        return url[36:55]

In [5]:
def Use_API_CVE(cve_ID):
    '''
    Renvoie un dictionnaire correspondant au CVE en question contenant:
    - la description
    - le score CVSS
    - le type CWE
    - la description du type CWE
    - la liste des produits affectés. Chaque élément de la liste est un dictionnaire contenant le nom du produit, le vendeur, et une liste des versions affectées.
    '''
    url = f"https://cveawg.mitre.org/api/cve/{cve_ID}" 
    response = requests.get(url) 
    data = response.json()

    CVE_dico = {}

    CVE_dico["description"] = data["containers"]["cna"]["descriptions"][0]["value"]

    #ATTENTION tous les CVE ne contiennent pas nécessairement ce champ, gérez l’exception,  
    #ou peut etre au lieu de cvssV3_0 c’est cvssV3_1 ou autre clé 

    try:
        CVE_dico["cvss"] = data["containers"]["cna"]["metrics"][0]["cvssV3_1"]["baseScore"]  
    except:
        CVE_dico["cvss"] = "Non disponible"
    

    CVE_dico["cwe"] = "Non disponible"
    CVE_dico["cwe_desc"] = "Non disponible"


    problemtype = data["containers"]["cna"].get("problemTypes", {})
    if problemtype and "descriptions" in problemtype[0]:
        CVE_dico["cwe"] = problemtype[0]["descriptions"][0].get("cweId", "Non disponible")
        CVE_dico["cwe_desc"] = problemtype[0]["descriptions"][0].get("description", "Non disponible")

    # Extraire les produits affectés
    i=0
    product_list = []
    affected = data["containers"]["cna"]["affected"]
    for product in affected:
        product_list.append({})
        product_list[i]["product"] = product["product"]
        product_list[i]["vendor"] = product["vendor"]
        product_list[i]["versions"] = [v["version"] for v in product["versions"] if v["status"] == "affected"]
        i += 1

    CVE_dico["affected_products"] = product_list

    return CVE_dico

In [6]:
def get_new_infos(url, ids_connus):
    '''
    Renvoie une liste de item
    '''
    RSS_page = get_RSS(url)
    list_new = []
    for i in range(len(RSS_page.entries)):
        entry = RSS_page.entries[-1 - i]          
        id_anssi = get_ID_ANSSI_from_link(entry.link)
        if id_anssi in ids_connus:
            break                                 
        list_new.append(entry)
    return list_new

In [7]:
def use_API_EPSS(cve_ID):
    url = f"https://api.first.org/data/v1/epss?cve={cve_ID}" 
    # Requête GET pour récupérer les données JSON 
    response = requests.get(url) 
    data = response.json() 
    # Extraire le score EPSS 
    epss_data = data.get("data", []) 
    if epss_data: 
        return epss_data[0]["epss"]
    else: 
        return "Non disponible"

In [8]:
# FOnction pour ajouter un niveau de dangerosité en fonction du score CVSS
def severite_depuis_cvss(score):
    if score == "Non disponible":
        return "Non disponible"
    try:
        s = float(score)
    except (ValueError, TypeError):
        return "Non disponible"

    if s >= 9:
        return "Critique"
    elif s >= 7:
        return "Élevée"
    elif s >= 4:
        return "Moyenne"
    else:
        return "Faible"

In [9]:
#Fonction qui va donc renvoyer une liste de lignes avec une ligne = un CVE
def construire_lignes_bulletin(entry, type_bulletin):
    lignes = []
    id_anssi = get_ID_ANSSI_from_link(entry.link)
    cve_list = get_CVE_list(entry.link)

    for cve in cve_list:
        try:
            infos = Use_API_CVE(cve)
            epss = use_API_EPSS(cve)
        except Exception as e:
            print(f"Erreur sur {cve} : {e}")
            continue
        finally:
            time.sleep(2)   # ça met une limite de temps, c'est demandé dans le projet donc j'ai mis

        produits = infos["affected_products"]
        editeurs = "; ".join(p["vendor"] for p in produits) or "Non disponible"
        noms     = "; ".join(p["product"] for p in produits) or "Non disponible"
        versions = "; ".join(", ".join(p["versions"]) for p in produits) or "Non disponible"

        lignes.append({
            "ID ANSSI": id_anssi,
            "Titre ANSSI": entry.title,
            "Type": type_bulletin,
            "Date": entry.get("published", "Non disponible"),
            "CVE": cve,
            "CVSS": infos["cvss"],
            "Base Severity": severite_depuis_cvss(infos["cvss"]),
            "CWE": infos["cwe"],
            "EPSS": epss,
            "Lien": entry.link,
            "Description": infos["description"],
            "Éditeur": editeurs,
            "Produit": noms,
            "Versions affectées": versions,
        })

    return lignes

In [10]:
#On a besoin de cette fonction pour plein de trucs en fait et c'est pas si dur après réflexion donc je l'ai fait
def type_depuis_url(url):
    if "avis" in url.lower():
        return "Avis"
    elif "alerte" in url.lower():
        return "Alerte"
    else:
        return "Inconnu"

In [11]:
#La fonction qu'on utilise pour traiter les nouvelles données
def traiter_flux(url, ids_connus=None):
    type_bulletin = type_depuis_url(url)
    if ids_connus is None:
        entries = get_RSS(url).entries      
    else:
        entries = get_new_infos(url, ids_connus) 
    lignes = []
    for entry in entries:
        lignes.extend(construire_lignes_bulletin(entry, type_bulletin))
    return lignes

In [12]:

#Ce serait la fonction qu'on utilise que 1 fois au début pour créer le dataFrame
def construire_dataframe(liste_urls):
    lignes = []
    for url in liste_urls:
        lignes.extend(traiter_flux(url))
    return pd.DataFrame(lignes)

In [13]:
def sauvegarder_csv(df, chemin="donnees_consolidees.csv"):
    df.to_csv(chemin, index=False, encoding="utf-8-sig")  

In [14]:
def charger_csv(chemin="donnees_consolidees.csv"):
    return pd.read_csv(chemin)

In [18]:
# --- TEST de bout en bout : tes vraies fonctions + sauvegarde CSV ---

avis   = get_RSS("https://www.cert.ssi.gouv.fr/avis/feed/")
alerte = get_RSS("https://www.cert.ssi.gouv.fr/alerte/feed/")

lignes = []
lignes += construire_lignes_bulletin(avis.entries[0], "Avis")
lignes += construire_lignes_bulletin(alerte.entries[0], "Alerte")

df = pd.DataFrame(lignes)
sauvegarder_csv(df)                          
print(f" {len(df)} lignes enregistrées")

print("Dossier de travail :", os.getcwd())
print("Fichier créé :", os.path.exists("donnees_consolidees.csv"))
print("Chemin complet :", os.path.abspath("donnees_consolidees.csv"))


 2 lignes enregistrées
Dossier de travail : c:\Users\lilia\Downloads
Fichier créé : True
Chemin complet : c:\Users\lilia\Downloads\donnees_consolidees.csv


In [ ]:
#####################################
#TEST pour voir, très long + de 20 minutes, j'ai meme pas oser finir
#####################################
urls = ["https://www.cert.ssi.gouv.fr/avis/feed/",
        "https://www.cert.ssi.gouv.fr/alerte/feed/"]

df = construire_dataframe(urls)
sauvegarder_csv(df)        
print(f"{len(df)} lignes récupérées")
df

NameError: name 'construire_dataframe' is not defined